# Amparo -- Baseline y fine-tuning LoRA del agente legal

Este notebook:
1. Carga `dataset_legal.jsonl` (1320 ejemplos en 24 categorias) y separa una validacion estratificada por categoria que **nunca** se usa para entrenar.
2. Corre el modelo base sin modificar sobre esa validacion (**baseline**).
3. Entrena un adaptador **LoRA** sobre el resto del dataset.
4. Vuelve a correr el modelo ya afinado sobre la misma validacion.
5. Compara baseline vs. afinado -- esa comparacion es la evidencia real de que el fine-tuning mejoro algo, no solo una suposicion.

Antes de correr: `Entorno de ejecucion > Cambiar tipo de entorno de ejecucion > GPU` (una T4 gratis alcanza usando 4-bit/QLoRA).

**Nota sobre las librerias:** `transformers`, `peft` y `trl` cambian su API con cierta frecuencia. Este notebook esta escrito contra las versiones estables mas recientes conocidas al momento de escribirlo; si algun cambio de firma da error, revisa el changelog de la libreria correspondiente -- la logica de cada celda (que hace y por que) sigue siendo valida aunque cambie algun nombre de parametro.

In [1]:
!pip install -q -U transformers peft trl bitsandbytes accelerate datasets wandb
!pip install -U "bitsandbytes>=0.46.1"

## Configuracion

Cambia `MODEL_ID` para comparar Qwen2.5 7B vs. Llama 3.1 8B (o el que haya ganado en el comparador local `tools/model_comparator`).

In [2]:
MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"  # alternativa: "meta-llama/Llama-3.1-8B-Instruct"

VAL_FRACTION = 0.15   # % por categoria reservado para validacion (nunca se entrena con esto)
RANDOM_SEED = 42

LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
LORA_TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj"]

NUM_EPOCHS = 3
LEARNING_RATE = 2e-4
PER_DEVICE_BATCH_SIZE = 2
GRAD_ACCUM_STEPS = 8   # batch efectivo = PER_DEVICE_BATCH_SIZE * GRAD_ACCUM_STEPS
MAX_SEQ_LENGTH = 1024

MAX_NEW_TOKENS_EVAL = 300
OUTPUT_DIR = "/content/amparo-lora"

## Monta tu Google Drive y ubica el dataset

Sube `dataset_legal.jsonl` una sola vez a `MyDrive/Colab Notebooks/Amparo/dataset_legal.jsonl` (ajusta `DATASET_PATH` si usaste otra ruta). Con Drive no tienes que volver a subir el archivo cada vez que el entorno de Colab se reinicia.

In [3]:
from google.colab import drive

drive.mount('/content/drive')

DATASET_PATH = '/content/drive/MyDrive/Colab Notebooks/Amparo/dataset_legal.jsonl'  # ajusta si usaste otra ruta
print(f'Usando dataset: {DATASET_PATH}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Usando dataset: /content/drive/MyDrive/Colab Notebooks/Amparo/dataset_legal.jsonl


## Cargar y separar train / validacion (estratificado por categoria)

In [4]:
import json
import random
from collections import defaultdict

random.seed(RANDOM_SEED)

records = [json.loads(line) for line in open(DATASET_PATH, encoding='utf-8')]

by_category = defaultdict(list)
for r in records:
    by_category[r['category']].append(r)

train_records, val_records = [], []
for category, items in by_category.items():
    items = items[:]
    random.shuffle(items)
    n_val = max(1, round(len(items) * VAL_FRACTION))
    val_records.extend(items[:n_val])
    train_records.extend(items[n_val:])

random.shuffle(train_records)
random.shuffle(val_records)

print(f'Total: {len(records)} | Train: {len(train_records)} | Validacion: {len(val_records)}')
for category in sorted(by_category):
    n_val = sum(1 for r in val_records if r['category'] == category)
    n_train = sum(1 for r in train_records if r['category'] == category)
    print(f'  {category:45s} train={n_train:3d}  val={n_val:3d}')

Total: 1320 | Train: 1119 | Validacion: 201
  Acceso a informacion publica                  train= 43  val=  8
  Accidentes de transito                        train= 52  val=  9
  Arriendo                                      train= 52  val=  9
  Comparendos de transito                       train= 52  val=  9
  Conciliacion prejudicial                      train= 43  val=  8
  Contratacion estatal y facturacion            train= 43  val=  8
  Contratos empresariales (B2B)                 train= 43  val=  8
  Derecho administrativo general                train= 43  val=  8
  Derecho ambiental sancionatorio               train= 43  val=  8
  Derecho contractual general                   train= 43  val=  8
  Derecho de familia - alimentos                train= 43  val=  8
  Despido                                       train= 53  val=  9
  Educacion / debido proceso disciplinario      train= 43  val=  8
  Embargos                                      train= 52  val=  9
  Garantias de con

## Cargar el modelo base en 4-bit (QLoRA)

Se carga cuantizado en 4-bit para que quepa comodamente en una GPU gratuita de Colab (T4, ~16GB).

In [5]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto',
)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

## Metrica de similitud

Misma heuristica lexica (difflib) que usa el comparador local en `tools/model_comparator/metrics.py`, para que los resultados sean comparables entre ambas herramientas. No es una metrica juridica rigurosa de correccion legal, solo sirve para ordenar/comparar respuestas de forma consistente.

In [6]:
from difflib import SequenceMatcher

SYSTEM_PROMPT = records[0]['messages'][0]['content']


def similarity_pct(expected: str, actual: str) -> float:
    if not expected or not actual:
        return 0.0
    ratio = SequenceMatcher(None, expected.strip().lower(), actual.strip().lower()).ratio()
    return round(ratio * 100, 1)


@torch.no_grad()
def generate_response(model, query: str) -> str:
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': query},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    output = model.generate(
        **inputs,
        max_new_tokens=MAX_NEW_TOKENS_EVAL,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
    )
    generated = output[0][inputs['input_ids'].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()


def evaluate(model, val_set, label: str):
    results = []
    for i, item in enumerate(val_set):
        query = item['messages'][1]['content']
        expected = item['messages'][2]['content']
        actual = generate_response(model, query)
        sim = similarity_pct(expected, actual)
        results.append({**item, 'generated': actual, 'similarity': sim})
        print(f"[{label}] {i + 1}/{len(val_set)}  sim={sim:5.1f}  {item['category']}")
    avg = sum(r['similarity'] for r in results) / len(results)
    print(f'\n[{label}] Similitud promedio: {avg:.1f}')
    return results

## Paso 1 -- Baseline (modelo base, sin fine-tuning)

Esto puede tardar varios minutos segun el tamano de la validacion.

In [7]:
baseline_results = evaluate(model, val_records, label='baseline')

with open('/content/baseline_results.jsonl', 'w', encoding='utf-8') as f:
    for r in baseline_results:
        f.write(json.dumps(r, ensure_ascii=False) + '\n')

[baseline] 1/201  sim=  1.9  Derecho de familia - alimentos
[baseline] 2/201  sim=  2.2  Propiedad y linderos
[baseline] 3/201  sim=  2.1  Salud / EPS
[baseline] 4/201  sim=  1.8  Educacion / debido proceso disciplinario
[baseline] 5/201  sim=  4.5  Accidentes de transito
[baseline] 6/201  sim=  2.6  Garantias de consumo
[baseline] 7/201  sim=  2.6  Relaciones laborales
[baseline] 8/201  sim=  2.4  Salud / EPS
[baseline] 9/201  sim= 13.3  Licencias urbanisticas
[baseline] 10/201  sim=  2.5  Conciliacion prejudicial
[baseline] 11/201  sim=  4.1  Derecho administrativo general
[baseline] 12/201  sim=  7.9  Licencias urbanisticas
[baseline] 13/201  sim=  7.3  Despido
[baseline] 14/201  sim=  4.6  Educacion / debido proceso disciplinario
[baseline] 15/201  sim=  2.8  Relaciones laborales
[baseline] 16/201  sim=  6.5  Derecho administrativo general
[baseline] 17/201  sim=  8.1  Salud / EPS
[baseline] 18/201  sim=  7.8  Contratos empresariales (B2B)
[baseline] 19/201  sim=  2.1  Pensiones y 

### (Opcional) Ya tienes un adaptador entrenado en Drive y no quieres reentrenar

Si esta es una sesion nueva y solo necesitas reproducir la evaluacion (Paso 3) sin gastar tiempo/cuota de GPU reentrenando, corre la celda de abajo en vez del Paso 2 completo, y luego salta directamente a la celda que desactiva gradient checkpointing (antes del Paso 3). La generacion es determinista (), asi que reproduce exactamente los mismos resultados que la corrida original.

In [ ]:
from peft import PeftModel

DRIVE_ADAPTER_DIR = '/content/drive/MyDrive/Colab Notebooks/Amparo/amparo-lora-adapter'
model = PeftModel.from_pretrained(model, DRIVE_ADAPTER_DIR)
print(f'Adaptador cargado desde {DRIVE_ADAPTER_DIR}')

## Paso 2 -- Fine-tuning con LoRA

### Antes de entrenar: conecta Weights & Biases

M1 pide dejar registrada la curva de perdida del entrenamiento como evidencia visual de que el modelo esta aprendiendo. Crea una cuenta gratuita en wandb.ai si no tienes una, y ten a mano tu API key (la consigues en https://wandb.ai/authorize).

In [8]:
import wandb

wandb.login()  # pega tu API key cuando te la pida (solo la primera vez)

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results


wandb: Enter your choice: 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.


wandb: Paste your API key and hit enter: ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: tomasposada67 (tomasposada67-universidad-eafit) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [9]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=LORA_TARGET_MODULES,
    task_type='CAUSAL_LM',
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 10,092,544 || all params: 7,625,709,056 || trainable%: 0.1323


In [10]:
from datasets import Dataset


def format_for_training(record):
    return {'text': tokenizer.apply_chat_template(record['messages'], tokenize=False)}


train_dataset = Dataset.from_list(train_records).map(format_for_training)

Map:   0%|          | 0/1119 [00:00<?, ? examples/s]

In [11]:
wandb.init(
    project='amparo-legal-finetune',
    name=f"lora-{MODEL_ID.split('/')[-1]}",
    config={
        'model_id': MODEL_ID,
        'lora_r': LORA_R,
        'lora_alpha': LORA_ALPHA,
        'epochs': NUM_EPOCHS,
        'learning_rate': LEARNING_RATE,
    },
)

from trl import SFTTrainer, SFTConfig

sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=PER_DEVICE_BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    learning_rate=LEARNING_RATE,
    logging_steps=10,
    save_strategy='epoch',
    bf16=True,
    dataset_text_field='text',
    max_length=MAX_SEQ_LENGTH,
    report_to='wandb',
    run_name=f"lora-{MODEL_ID.split('/')[-1]}",
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_dataset,
    processing_class=tokenizer,
)

trainer.train()

print('Dashboard de W&B (guarda este link para tu informe de M1):', wandb.run.url)
wandb.finish()

Tokenizing train dataset:   0%|          | 0/1119 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/1119 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/1119 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/1119 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
10,2.419211
20,1.146324
30,0.941670
40,0.905339
50,0.873919
60,0.845621
70,0.859781
80,0.790317
90,0.775341
100,0.798977


Dashboard de W&B (guarda este link para tu informe de M1): https://wandb.ai/tomasposada67-universidad-eafit/amparo-legal-finetune/runs/f1g0r0ra


train/entropy,█▅▃▃▃▂▂▂▂▂▂▁▂▁▁▁▁▁▁▁▁
train/epoch,▁▁▂▂▂▃▃▃▄▄▄▅▅▆▆▆▇▇▇███
train/global_step,▁▁▂▂▂▃▃▃▄▄▅▅▅▆▆▆▇▇▇███
train/grad_norm,█▁▂▁▁▂▂▂▂▂▃▃▂▃▂▃▃▃▃▃▃
train/learning_rate,██▇▇▇▆▆▆▅▅▄▄▄▃▃▃▂▂▂▁▁
train/loss,█▃▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/mean_token_accuracy,▁▆▇▇▇▇▇▇█▇███████████
train/num_tokens,▁▁▂▂▂▃▃▃▄▄▄▅▅▆▆▆▇▇▇██
total_flos,2.075352344980992e+16
train/entropy,0.72674
train/epoch,3


## Guardar el adaptador LoRA en Drive

Se guarda directo en tu Drive (no solo son unos MB, son los pesos LoRA) para que no se pierda si el entorno de Colab se desconecta a mitad del entrenamiento o despues.

In [12]:
import shutil

model.save_pretrained(f'{OUTPUT_DIR}/adapter')
tokenizer.save_pretrained(f'{OUTPUT_DIR}/adapter')

DRIVE_ADAPTER_DIR = '/content/drive/MyDrive/Colab Notebooks/Amparo/amparo-lora-adapter'
shutil.copytree(f'{OUTPUT_DIR}/adapter', DRIVE_ADAPTER_DIR, dirs_exist_ok=True)
print(f'Adaptador guardado en {DRIVE_ADAPTER_DIR}')

Adaptador guardado en /content/drive/MyDrive/Colab Notebooks/Amparo/amparo-lora-adapter


In [13]:
# prepare_model_for_kbit_training() dejo gradient checkpointing activo, lo que fuerza use_cache=False
# durante model.generate() (recalcula toda la atencion en cada token, sin KV-cache). Sin esto, el
# Paso 3 tarda varias veces mas de lo necesario y puede agotar la cuota de GPU de Colab a mitad de la evaluacion.
model.gradient_checkpointing_disable()
model.config.use_cache = True
model.eval()

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen2ForCausalLM(
      (model): Qwen2Model(
        (embed_tokens): Embedding(152064, 3584)
        (layers): ModuleList(
          (0-27): 28 x Qwen2DecoderLayer(
            (self_attn): Qwen2Attention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=3584, out_features=3584, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=3584, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=3584, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora

## Paso 3 -- Evaluar el modelo ya afinado (misma validacion)

In [14]:
finetuned_results = evaluate(model, val_records, label='fine-tuned')

with open('/content/finetuned_results.jsonl', 'w', encoding='utf-8') as f:
    for r in finetuned_results:
        f.write(json.dumps(r, ensure_ascii=False) + '\n')

[fine-tuned] 1/201  sim=  1.9  Derecho de familia - alimentos
[fine-tuned] 2/201  sim= 10.1  Propiedad y linderos
[fine-tuned] 3/201  sim= 50.4  Salud / EPS
[fine-tuned] 4/201  sim= 25.1  Educacion / debido proceso disciplinario
[fine-tuned] 5/201  sim= 15.8  Accidentes de transito
[fine-tuned] 6/201  sim= 30.7  Garantias de consumo
[fine-tuned] 7/201  sim=  5.9  Relaciones laborales
[fine-tuned] 8/201  sim= 45.4  Salud / EPS
[fine-tuned] 9/201  sim= 19.7  Licencias urbanisticas
[fine-tuned] 10/201  sim=  4.0  Conciliacion prejudicial
[fine-tuned] 11/201  sim= 48.7  Derecho administrativo general
[fine-tuned] 12/201  sim= 12.3  Licencias urbanisticas
[fine-tuned] 13/201  sim=  9.0  Despido
[fine-tuned] 14/201  sim=  5.7  Educacion / debido proceso disciplinario
[fine-tuned] 15/201  sim= 14.3  Relaciones laborales
[fine-tuned] 16/201  sim= 32.4  Derecho administrativo general
[fine-tuned] 17/201  sim=  4.5  Salud / EPS
[fine-tuned] 18/201  sim= 19.6  Contratos empresariales (B2B)
[fine-

## Paso 4 -- Comparacion baseline vs. afinado

Esta tabla es la evidencia real (no una suposicion) de si el fine-tuning mejoro las respuestas, por categoria y en promedio general.

In [15]:
import pandas as pd

df_base = pd.DataFrame(baseline_results)[['id', 'category', 'similarity']].rename(columns={'similarity': 'baseline'})
df_ft = pd.DataFrame(finetuned_results)[['id', 'similarity']].rename(columns={'similarity': 'fine_tuned'})
comparison = df_base.merge(df_ft, on='id')
comparison['mejora'] = comparison['fine_tuned'] - comparison['baseline']

summary = comparison.groupby('category')[['baseline', 'fine_tuned', 'mejora']].mean().round(1)
print(summary)
print(
    f"\nPromedio general -> baseline: {comparison['baseline'].mean():.1f}  "
    f"fine-tuned: {comparison['fine_tuned'].mean():.1f}  "
    f"mejora: {comparison['mejora'].mean():+.1f}"
)

comparison.to_csv('/content/comparacion_baseline_vs_finetuned.csv', index=False)

                                          baseline  fine_tuned  mejora
category                                                              
Acceso a informacion publica                   2.5        10.9     8.4
Accidentes de transito                         3.2        12.3     9.0
Arriendo                                       3.9        11.2     7.2
Comparendos de transito                        3.3        29.2    25.9
Conciliacion prejudicial                       4.1        14.9    10.8
Contratacion estatal y facturacion             2.6        19.4    16.8
Contratos empresariales (B2B)                  4.5        21.2    16.7
Derecho administrativo general                 3.9        24.0    20.1
Derecho ambiental sancionatorio                2.9        19.6    16.6
Derecho contractual general                    3.8        22.4    18.6
Derecho de familia - alimentos                 2.8        10.1     7.2
Despido                                        3.2        14.0    10.8
Educac

In [ ]:
# Los resultados por ejemplo (pregunta/esperado/generado/similitud) solo viven en el almacenamiento
# efimero de /content -- se pierden si la sesion se desconecta. Se persisten en Drive junto al adaptador.
DRIVE_DIR = '/content/drive/MyDrive/Colab Notebooks/Amparo'
shutil.copy('/content/baseline_results.jsonl', f'{DRIVE_DIR}/baseline_results.jsonl')
shutil.copy('/content/finetuned_results.jsonl', f'{DRIVE_DIR}/finetuned_results.jsonl')
shutil.copy('/content/comparacion_baseline_vs_finetuned.csv', f'{DRIVE_DIR}/comparacion_baseline_vs_finetuned.csv')
print(f'Resultados guardados en {DRIVE_DIR}')